# Introduction to JACC.jl
<img src="../images/JACC-logo.png" alt="JACC logo" style="width:15%;height:auto;">

- [Motivation](#Motivation)
- [Getting started with JACC](#Getting-started-with-JACC)
- [JACC paradigm basics](#JACC-paradigm-basics)
- [API summaries](#API-summaries)

## Motivation
Different HPC systems have different hardware and thus different vendor-provided APIs.

Different users have different levels/types of expertise.

Therefore we want:
- performance portability
- programming productivity
- performance "practicability"?

<img src="../images/HPC-systems-vendors.png" alt="HPC Systems and Vendors" style="width:90%;height:auto;">

### Options for (data) parallelism in Julia
CPU multi-threading
- Threads (built-in)
- Polyester
- OhMyThreads

GPU kernel model (fine-granularity):
- Vendor-specific packages:
  - CUDA, AMDGPU, oneAPI, Metal
- [KernelAbstractions](https://github.com/JuliaGPU/KernelAbstractions.jl)
  - Provides (mostly) portable interface implemented by each vendor package

### The Aim of JACC

Provide a familiar interface and smart defaults, bringing good performance closer to domain scientists who may not be experts in computer science.

- high-level unified Julia front-end on top of multiple backends (Threads, CUDA, AMDGPU, oneAPI)
- Hide low-level and device-specific implementation
- Improve programming productivity for science and HPC

<img src="../images/JACC-stack.png" alt="JACC stack" style="width:40%;height:auto;">

Reach out on [github](https://github.com/JuliaORNL/JACC.jl/discussions)

#### Performance results for simple kernels
<details>
    <summary>Image</summary>
<img src="../images/JACC-perfresults-01.png" alt="Performance Results" style="width:90%;height:auto;">
</details>

## Simple Exercises
1. Allocate an array and use a `parallel_for` to initialize the elements (e.g., using the index values)
2. Allocate an array of ones and find the sum of the elements using `parallel_reduce`.
3. Try #2 using a parallel_for with `@atomic`

(Bonus) Create a struct holding an array and use it in a kernel. (This is not a challenge for CPU-only code)

## Getting started with JACC

##### 1. Add JACC as a dependency

In [ ]:
import Pkg
Pkg.add("JACC")
Pkg.status()

##### 2. Set backend

The default backend is Threads

In [ ]:
import JACC
@show JACC.backend
@show JACC.supported_backends
@show names(JACC.Backend, all = true)
;

Calling `set_backend` will install the appropriate backend package as a dependency for the current project (don't commit this!).

NOTE: in some cases you may need to configure the backend package before proceeding to use it.

In [ ]:
import JACC
JACC.set_backend("oneapi")
# JACC.set_backend(:oneapi)
# JACC.set_backend(JACC.Backend.oneapi)

##### 3. Restart Julia
For this notebook, use the menu: "Kernel" > "Restart Kernel..." (With the REPL in a terminal you would simply quit and restart `julia`.)

##### 4. Load extension before use

```julia
JACC.@init_backend
```

Equivalent to
```julia
import <backend-package>
```

In [ ]:
import JACC
JACC.@init_backend # import oneAPI

## JACC paradigm basics

### Arrays
There is no separate array type in JACC. Array functions return the backend array type directly.

`JACC.array_type()` will give the corresponding `UnionAll` (non-instantuiated) parametric type.
- useful for function parameters and struct members.

In [ ]:
@show JACC.array_type()
@show JACC.array_type(){Float64, 2}
;

#### Create arrays on-device

- `JACC.ones`
- `JACC.zeros`
- `JACC.fill`
- `JACC.array`

Without a type parameter, these use the default float type (usually `Float64`), obtainable via `JACC.default_float()`.

In [ ]:
const JFloat = JACC.default_float()
@show JFloat
;

In [ ]:
ad_ones = JACC.ones(4)
println("\n'JACC.ones(4)'")
display(ad_ones)

ad_zeros = JACC.zeros(3, 3)
println("\n'JACC.zeros(3, 3)'")
display(ad_zeros)

ad_fill = JACC.fill(2.5, 4, 2)
println("\n'JACC.fill(2.5, 4, 2)'")
display(ad_fill)

ad_array = JACC.array(3, 2) # uninitialized
println("\n'JACC.array(3, 2)' => ad_array:")
display(ad_array)
;

### Transfer

Make a copy

- `JACC.to_device` or `JACC.array(::AbstractArray)`
- `JACC.to_host`

**NOTE**: For `Threads` backend, the above return their argument.

Copy data only (already allocated)

- `JACC.transfer!(dst, src)`

In [ ]:
println("'JACC.to_host( JACC.fill(2.5, 4, 2) )'")
ah_fill = JACC.to_host(ad_fill)
display(ah_fill)

println("\n'randn(3, 2)' => ah_rand:")
ah_rand = randn(3, 2)
display(ah_rand)

println("\n'JACC.to_device( randn(3, 2) )'")
ad_rand = JACC.to_device(ah_rand)
display(ad_rand)

println("\nah_rand after 'JACC.transfer!(ah_rand, ad_array)'")
JACC.transfer!(ah_rand, ad_array)
display(ah_rand)
;

### Parallel Kernel Constructs

Primary components:
- kernel workload (size(s) or range(s))
- kernel function
- parameters for kernel function

**NOTE**: kernel function must take thread index(es) as fist parameter(s).

```julia
function f(i, x)
    #= 
    ... 
    =#
end
```

#### From serial loop to `parallel_for`

```julia
### Serial loop
for i = 1:N
    y[i] += α * x[i]
end
###
```
- workload: `N`
- function: `y[i] += α * x[i]`
- parameters: `α, x, y`
```julia
### JACC do-style
JACC.parallel_for(N, α, xd, yd) do i, a, x, y
    y[i] += a * x[i]
end
###
```

#### Function style
- `JACC.parallel_for(N, f, x)`
- `JACC.parallel_reduce(N, f, x)`

#### Macro style
- `JACC.@parallel_for range=N f(x)`
- `JACC.@parallel_reduce range=N f(x)`

**NOTE**: keyword arguments in macro calls cannot use spaces

Example: define `axpy` and `dot` as kernel functions for 1D and 2D arrays:

In [ ]:
# 1D - with classic call style

## construct arrays
SIZE = 1_000
x = round.(rand(JFloat, SIZE) * 100)
y = round.(rand(JFloat, SIZE) * 100)
α = 2.5

xd = JACC.to_device(x)
yd = JACC.to_device(y)

## perform axpy
function axpy(i, α, x, y) # element-wise kernel for y = αx + y
    y[i] += α * x[i]
end

JACC.parallel_for(SIZE, axpy, α, xd, yd)

## perform dot product
function dot(i, x, y)     # element-wise kernel for x ⋅ y
    return x[i] * y[i]
end

res = JACC.parallel_reduce(SIZE, dot, xd, yd)

@show res
;

In [ ]:
# 2D - with macro style

## construct arrays
X = round.(rand(JFloat, SIZE, SIZE) * 100)
Y = round.(rand(JFloat, SIZE, SIZE) * 100)

Xd = JACC.to_device(X)
Yd = JACC.to_device(Y)

## perform axpy
function axpy(i, j, α, x, y)
    y[i, j] += α * x[i, j]
end

JACC.@parallel_for range=(SIZE, SIZE) axpy(α, Xd, Yd) # cannot use spaces for keyword arguments

## perform dot product
function dot(i, j, x, y)
    return x[i, j] * y[i, j]
end

ret = JACC.@parallel_reduce range=(SIZE, SIZE) dot(Xd, Yd)

# NOTE: macro @parallel_reduce returns an on-device array with one element
res = JACC.to_host(ret)[]

@show ret
@show res
;

#### Julia `do`-block syntactic sugar

When a function's first parameter is a function, that argument can be skipped and defined in-place using `do ... end`.

Manual [section](https://docs.julialang.org/en/v1/manual/functions/#Do-Block-Syntax-for-Function-Arguments)
uses `map` as example:

```julia
map(f, A)
```

where `f` is a unary function of `f(x)`

```julia
map(A) do x
    # function body defined in-place
end
```

Remember, in JACC, a kernel function must take an index in addition to the other expected parameters.

In [ ]:
### JACC do-style

JACC.parallel_for(SIZE, α, xd, yd) do i, a, x, y
    x[i] += a * y[i]
end

dp = JACC.parallel_reduce(SIZE, xd, yd) do i, x, y
    return x[i] * y[i]
end
;

### [Aside]: kernel arguments

Items must be passed explicitly. 

An implicit reference to a name from the parent scope will be attempting to access host memory from the device (unless you're using the "threads" backend).

Let's see a slightly more involved example and use a named tuple for arguments.

```julia
### Serial loop
for i = 1:length(events)
    @inbounds begin
        event = events[i, 6:8]
        weight = events[i, 1]
        for op in transforms
            v = op * event
            atomic_push!(hist, v, weight)
        end
    end
end

### JACC do-style
JACC.parallel_for(   # named tuple
    length(events), (; hist, events, transforms)) do i, t
    @inbounds begin
        event = t.events[i, 6:8]
        weight = t.events[i, 1]
        for op in t.transforms
            v = op * event
            atomic_push!(t.hist, v, weight)
        end
    end
end
###

### JACC with anonymous function
JACC.parallel_for(length(events),
    (i, t) -> begin
        @inbounds begin
            event = t.events[i, 6:8]
            weight = t.events[i, 1]
            for op in t.transforms
                v = op * event
                atomic_push!(t.hist, v, weight)
            end
        end
    end,
    (; hist, events, transforms), # named tuple
)
###
```

## API summaries

### `JACC.parallel_for`

```julia
# idiomatic function application
parallel_for(f, range, x...; [name])

# classic version
parallel_for(range, f, x...; [name])

# keyword arguments version
parallel_for(; range, f, args, name) # args must be a Tuple

# macro version
@parallel_for range=range [name=...] f(x...)

```

The `range` can be one of
- `Integer`
- `AbstractRange`
- `NTuple{N, Integer}`
- `NTuple{N, AbstractRange}`

In [ ]:
let
    a = JACC.zeros(Int32, 10, 10)
    JACC.parallel_for((2:9, 4:9), a) do i, j, a
        a[i, j] += i * j
    end
    display(a)
end

### `JACC.parallel_reduce`

```julia
### idiomatic function application
#
# return: typeof(init)
# defaults:
#  - type: JACC.default_float()
#  - op: +
#  - init: default_init(type, op) # 0.0 for +
parallel_reduce(f, range, x...; [type], [op], [init], [name])

### classic version
parallel_reduce(range, f, x...; [type], [op], [init], [name])

### keyword-arguments version
#
# return: "on-device" single-element array
parallel_reduce(; range, f, args, [type], [op], [init], [name]) # args must be a Tuple

### macro version
#
# return: "on-device" single-element array
@parallel_reduce range=range [type=...] [op=...] [init=...] [name=...] f(x...)

# op ∊ {+, *, min, max, <user-defined>}

### array-based reduce
#
# return: typeof(init)
# defaults:
#  - op: +
#  - init: default_init(eltype(a), op)
parallel_reduce([op, ] a::AbstractArray; [range], [init], [name])
# macro version on array
@parallel_reduce range=range [type=...] [op=...] [init=...] [name=...] a
```

Here are a few simple examples:

In [ ]:
a = JACC.array([i for i in 1:10])
@show JACC.parallel_reduce(a)
@show JACC.parallel_reduce(min, a)
;

In [ ]:
b = JACC.fill(2, 10)
@show b
@show eltype(b)
# Reduce to a different result type
b_min = JACC.parallel_reduce(6:10, b; op = min, type = Float32) do i, b
    b[i] + i
end
@show b_min
# Macro version on array
b_max = JACC.@parallel_reduce range=10 op=max init=-Inf b
@show JACC.to_host(b_max)[]
;

## Notes for GPU programming
- Any function can be used in a kernel (without special annotation) as long as it's thread-safe and doesn't allocate
- JACC imports the `@atomic` macro from [Atomix.jl](https://github.com/JuliaConcurrent/Atomix.jl)
- Use [StaticArrays.jl](https://github.com/JuliaArrays/StaticArrays.jl) for small fixed-size arrays
- Use [Adapt.jl](https://github.com/JuliaGPU/Adapt.jl) if you need to put GPU arrays in a struct

## Exercises

See exercises in "code" directory.

## Applications and More

Examples and applications using JACC
- <mark>https://github.com/JuliaORNL/JACC-applications</mark>
- https://github.com/JuliaORNL/GrayScott.jl
- https://github.com/tdehoff/JACC-7-point-stencil
- https://github.com/JuliaORNL/MiniVATES.jl

Extras
- `JACC.launch_spec` : provide kernel launch parameters
- `JACC.shared` : makes using on-chip shared memory easy
- `JACC.Multi` : distribute workload across available GPUs (single-node)
- `JACC.Async` : launch kernels asynchronously on multiple GPUs (single-node)

Ongoing Efforts
- `parallel_scan`... coming soon!
- Performance benchmarking (closing gaps)
- Multi-node kernel launching
- JACC.AI
- Have ideas? Send us a message!